# 05 Centerlines: OIB, bed and velocity profiles along Seward's centerline and transverse line

Samples the OIB elevation-difference rasters, the bed and the ITS_LIVE-corrected velocities along `Centerline_1` (centerline) and `Centerline_2` (transverse line); computes the zone means (Table 2), transverse zone means, the 2007->2009 sign changes and the transverse elevation-difference statistics; writes the profile arrays read by `figures/fig4_5_profiles.py` (Figs. 4-5).

Outputs: `../Data_Paper/tables/centerlines/`

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import pyproj
import xdem
from shapely.geometry import Point, LineString
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from tqdm import tqdm

OUT = '../Data_Paper/tables/centerlines'
os.makedirs(OUT, exist_ok=True)

## Sampling lines
10 m samples along the centerline and the transverse line; distance along each line (centerline from its terminus end, transverse line from its west end).

In [ ]:
def linestrings_to_array(gdf, sample_distance=100, invert=False):
    """
    Convert LineString geometries to a 2D array with X, Y coordinates and LineString ID,
    sampling points at regular intervals along each line.
    Returns numpy array with shape (n_points, 3) where columns are [X, Y, ID]
    """
    points_data = []
    for idx, row in gdf.iterrows():
        linestring = row['geometry']
        line_id = row['id']
        if isinstance(linestring, LineString):
            total_length = linestring.length
            num_samples = int(total_length / sample_distance) + 1
            for i in range(num_samples):
                distance = i * sample_distance
                if distance > total_length:
                    distance = total_length
                point = linestring.interpolate(distance)
                points_data.append([point.x, point.y, line_id])
                if distance >= total_length:
                    break
    points_array = np.array(points_data)
    # Invert the rows to have points from start to end
    if invert == True:
        points_array = points_array[::-1]
    return points_array

In [ ]:
gdf = gpd.read_file('../Data_Paper/vectors/Centerline_1.shp')
gdf2 = gpd.read_file('../Data_Paper/vectors/Centerline_2.shp')

centerline = linestrings_to_array(gdf, sample_distance=10)
cross_line = linestrings_to_array(gdf2, sample_distance=10)
# Straight-line distance from the first point (old axis, kept to convert the old km markers)
centerline[1:,2] = np.sqrt(np.sum((centerline[1:, :2] - centerline[0, :2]) ** 2, axis=1))
centerline[0,2] = 0
cross_line[1:,2] = np.sqrt(np.sum((cross_line[1:, :2] - cross_line[0, :2]) ** 2, axis=1))
cross_line[0,2] = 0

# Distance ALONG the lines
_seg = np.hypot(np.diff(centerline[:, 0]), np.diff(centerline[:, 1]))
centerline_straight = centerline[:, 2].copy()
centerline[:, 2] = np.r_[0, np.cumsum(_seg)]
print(f"centerline length along line: {centerline[-1, 2] / 1e3:.2f} km (straight-line max was {centerline_straight.max() / 1e3:.2f} km)")

_seg_x = np.hypot(np.diff(cross_line[:, 0]), np.diff(cross_line[:, 1]))
cross_line_straight = cross_line[:, 2].copy()
cross_line[:, 2] = np.r_[0, np.cumsum(_seg_x)]
print(f"transverse line west end at x={cross_line[0,0]:.0f}, east end x={cross_line[-1,0]:.0f}; length along line {cross_line[-1, 2] / 1e3:.2f} km (straight {cross_line_straight.max() / 1e3:.2f} km)")

## OIB profiles and bed
Nearest valid OIB value within 200 m (500 m for 2021-05-02), Savitzky-Golay filter, change relative to 1995.

In [ ]:
def find_closest_points_corrected(px, py, x_valid, y_valid, z_valid, max_search_distance=None):
    """
    Closest non-NaN elevation value for each line point (NaN beyond max_search_distance),
    then NaN where the jump to the previous sample exceeds 5 std of the jumps.
    """
    centerline_points = np.column_stack([px, py])
    valid_dem_points = np.column_stack([x_valid, y_valid])
    tree = cKDTree(valid_dem_points)
    query_distances, indices = tree.query(centerline_points, k=1)
    z_found = z_valid[indices].copy()
    distances = query_distances.copy()
    if max_search_distance is not None:
        beyond_threshold = distances > max_search_distance
        z_found[beyond_threshold] = np.nan

    threshold = np.nanstd(np.abs(z_found[1:] - z_found[:-1]))*5
    large_diff = np.abs(z_found[1:] - z_found[:-1]) > threshold
    z_found[1:][large_diff] = np.nan

    return z_found, distances

In [ ]:
oib_list = [f for f in os.listdir('../Data_Paper/OIB_elevation_differences/') if f.split('_')[-1] == 'OIB.tif']
dates_oib = [pd.to_datetime(f.split('_')[0]) for f in os.listdir('../Data_Paper/OIB_elevation_differences/') if f.split('_')[-1] == 'OIB.tif']

# Sort by dates
inds_sorted = np.argsort(dates_oib)
oib_list = [oib_list[i] for i in inds_sorted]
dates_oib = [dates_oib[i] for i in inds_sorted]

dates_to_remove = [pd.to_datetime('2000-06-23'), pd.to_datetime('2012-03-18'), pd.to_datetime('2015-05-14'), pd.to_datetime('2016-05-28')]
indices_to_remove = [i for i, date in enumerate(dates_oib) if date in dates_to_remove]
oib_list = [oib_list[i] for i in range(len(oib_list)) if i not in indices_to_remove]
dates_oib = [dates_oib[i] for i in range(len(dates_oib)) if i not in indices_to_remove]

oib_95 = xdem.DEM(f'../Data_Paper/OIB_elevation_differences/{oib_list[0]}')

centerline_profiles = np.full(((len(oib_list), centerline.shape[0])), fill_value=np.nan)
cross_line_profiles = np.full(((len(oib_list), cross_line.shape[0])), fill_value=np.nan)

for i in tqdm(range(len(oib_list))):
    oib_dem = xdem.DEM(f"../Data_Paper/OIB_elevation_differences/{oib_list[i]}", nodata = np.nan).reproject(bounds=oib_95.bounds)
    oib_dem.data.mask[oib_dem.data < 0] = True  # Mask out negative values
    x_valid = oib_dem.coords()[0][~oib_dem.data.mask]
    y_valid = oib_dem.coords()[1][~oib_dem.data.mask]
    z_valid = oib_dem.data[~oib_dem.data.mask]

    radius = 500 if dates_oib[i] == pd.Timestamp('2021-05-02') else 200  # 2021 flew up to ~0.5 km off the line
    centerline_profiles[i], _ = find_closest_points_corrected(centerline[:,0], centerline[:,1], x_valid, y_valid, z_valid, max_search_distance=radius)
    cross_line_profiles[i], _ = find_closest_points_corrected(cross_line[:,0], cross_line[:,1], x_valid, y_valid, z_valid, max_search_distance=radius)

bed = xdem.DEM('../Data_Paper/bed/malaspina_bed_epsg3338.tif').reproject(crs=32607)
bed_centerline = bed.interp_points((centerline[:,0], centerline[:, 1]), method='nearest')
bed_cross_line = bed.interp_points((cross_line[:,0], cross_line[:, 1]), method='nearest')

In [ ]:
centerline_smoothed = np.array([savgol_filter(profile, 50, 2) for profile in centerline_profiles[:]])
cross_line_smoothed = np.array([savgol_filter(profile, 10, 2) for profile in cross_line_profiles[:]])

dhdt_centerline = np.array([(centerline_smoothed[i] - centerline_smoothed[0]) / (((dates_oib[i] - dates_oib[0]).days)/365.25) for i in range(0, len(dates_oib))])
dhdt_cross_line = np.array([(cross_line_smoothed[i] - cross_line_smoothed[0]) / (((dates_oib[i] - dates_oib[0]).days)/365.25) for i in range(0, len(dates_oib))])

In [ ]:
# Profiles shown: centerline coverage >= 70 % (raw samples within 200 m), plus 2021-05-02
coverage = np.isfinite(centerline_profiles).mean(axis=1)
show = [i for i in range(1, len(dates_oib)) if coverage[i] >= 0.70 or dates_oib[i] == pd.Timestamp('2021-05-02')]
for i, d in enumerate(dates_oib):
    print(f"{d.date()}  coverage {coverage[i] * 100:5.1f}%  {'shown' if i in show else ('reference' if i == 0 else 'dropped')}")

# Transverse line: centerline set + 2019-09-28; 2000-08-26 and 2009 kept although partial
coverage_x = np.isfinite(cross_line_profiles).mean(axis=1)
show_x = sorted(set(show) | {dates_oib.index(pd.Timestamp('2019-09-28'))})
for i, d in enumerate(dates_oib):
    print(f"transverse {d.date()}  coverage {coverage_x[i] * 100:5.1f}%  {'shown' if i in show_x else ''}")

## Velocities
Monthly medians of the corrected ITS_LIVE velocities (2000-2022) at 100 m samples along both lines; positions as distance along the UTM lines.

In [ ]:
gdf_vels = gdf.copy()
gdf_vels = gdf_vels.to_crs(epsg=3413)
centerline_vels = linestrings_to_array(gdf_vels, sample_distance=100)
centerline_vels[1:,2] = np.sqrt(np.sum((centerline_vels[1:, :2] - centerline_vels[0, :2]) ** 2, axis=1))

vx = xr.open_dataset(
    '../External_Data/velocity_corrected/vx_5D_10modes_1e4_correction.nc'
    ).sel(y=xr.DataArray(centerline_vels[:,1], dims='z'),
          x=xr.DataArray(centerline_vels[:,0], dims='z'),
          method='nearest'
    ).sel(mid_date=slice(
        np.datetime64('2000-01-01'), np.datetime64('2022-12-31')))

vy = xr.open_dataset(
    '../External_Data/velocity_corrected/vy_5D_10modes_1e4_correction.nc'
    ).sel(y=xr.DataArray(centerline_vels[:,1], dims='z'),
          x=xr.DataArray(centerline_vels[:,0], dims='z'),
          method='nearest'
    ).sel(mid_date=slice(
        np.datetime64('2000-01-01'), np.datetime64('2022-12-31')))

v_centerline = np.sqrt(vx.vx**2 + vy.vy**2)
v_centerline = v_centerline.resample(mid_date='1M').median(dim='mid_date')
v_centerline.values = v_centerline.where(v_centerline.values >= 0, np.nan)

# Transverse line
gdf_vels = gdf2.copy()
gdf_vels = gdf_vels.to_crs(epsg=3413)
crossline_vels = linestrings_to_array(gdf_vels, sample_distance=100)
crossline_vels[1:,2] = np.sqrt(np.sum((crossline_vels[1:, :2] - crossline_vels[0, :2]) ** 2, axis=1))

vx_cross = xr.open_dataset(
    '../External_Data/velocity_corrected/vx_5D_10modes_1e4_correction.nc'
    ).sel(y=xr.DataArray(crossline_vels[:,1], dims='z'),
          x=xr.DataArray(crossline_vels[:,0], dims='z'),
          method='nearest'
    ).sel(mid_date=slice(
        np.datetime64('2000-01-01'), np.datetime64('2022-12-31')))

vy_cross = xr.open_dataset(
    '../External_Data/velocity_corrected/vy_5D_10modes_1e4_correction.nc'
    ).sel(y=xr.DataArray(crossline_vels[:,1], dims='z'),
          x=xr.DataArray(crossline_vels[:,0], dims='z'),
          method='nearest'
    ).sel(mid_date=slice(
        np.datetime64('2000-01-01'), np.datetime64('2022-12-31')))

v_crossline = np.sqrt(vx_cross.vx**2 + vy_cross.vy**2)
v_crossline = v_crossline.resample(mid_date='1M').median(dim='mid_date')
v_crossline.values = v_crossline.where(v_crossline.values >= 0, np.nan)

In [ ]:
# Velocity sample positions as distance along the UTM lines (same lines, projected)
_line = gdf.to_crs(32607).geometry.iloc[0]
_tr = pyproj.Transformer.from_crs(3413, 32607, always_xy=True)
_px, _py = _tr.transform(centerline_vels[:, 0], centerline_vels[:, 1])
centerline_vels[:, 2] = [_line.project(Point(a, b)) for a, b in zip(_px, _py)]
assert abs(_line.length - centerline[-1, 2]) < 50, (_line.length, centerline[-1, 2])

_line_x = gdf2.to_crs(32607).geometry.iloc[0]
_px, _py = _tr.transform(crossline_vels[:, 0], crossline_vels[:, 1])
crossline_vels[:, 2] = [_line_x.project(Point(a, b)) for a, b in zip(_px, _py)]
assert abs(_line_x.length - cross_line[-1, 2]) < 50, (_line_x.length, cross_line[-1, 2])

## Markers, zone boundaries and km conversion
Numbered points (old straight-line km values converted to along-line km), zone shading boundaries. Writes `centerline_markers.csv`, `crossline_markers.csv`, `km_conversion.csv`.

In [ ]:
# Centerline: first marker = accumulation/trunk boundary (old 9.3 km); others = distances cited in the Results (old axis)
max_dist_km = centerline[:,2].max() / 1000
text_km_old = [centerline_straight.max() / 1000 - 70, 24, 33, 39, 58, 64, 65, 66, 75]
_old_x = centerline_straight.max() / 1000 - centerline_straight / 1000
poi_inds = [int(np.argmin(np.abs(_old_x - k))) for k in text_km_old]
poi = [centerline[j, 2] for j in poi_inds]  # along-line m from terminus end
marker_table = pd.DataFrame({'marker': range(1, len(poi) + 1), 'old_km': text_km_old,
                             'new_km': [round(centerline[-1, 2] / 1000 - p / 1000, 1) for p in poi],
                             'km': [max_dist_km - p / 1000 for p in poi], 'sample_index': poi_inds})
print(marker_table.to_string(index=False)); marker_table.to_csv(f'{OUT}/centerline_markers.csv', index=False)
ind_40 = np.where(centerline_straight < 40000)[0][-1]   # zone boundaries: same physical points as the old 40/70 km
ind_70 = np.where(centerline_straight < 70000)[0][-1]
b_lobe, b_trunk = centerline[ind_40, 2] / 1000, centerline[ind_70, 2] / 1000

# Transverse line: old straight-line 10, 25, 30 km, plus the western start of the bed mound (along-line 7.5 km)
text_km_old_x = [10, 25, 30]
poi_inds_x = [np.where(cross_line_straight < k * 1000)[0][-1] for k in text_km_old_x]
poi_inds_x = [int(np.argmin(np.abs(cross_line[:, 2] - 7500)))] + poi_inds_x
poi_x = [cross_line[j, 2] / 1000 for j in poi_inds_x]
marker_table_x = pd.DataFrame({'marker': range(1, len(poi_x) + 1), 'old_km': [None] + text_km_old_x, 'new_km': [round(p, 1) for p in poi_x],
                               'km': poi_x, 'sample_index': poi_inds_x})
print(marker_table_x.to_string(index=False)); marker_table_x.to_csv(f'{OUT}/crossline_markers.csv', index=False)
ind_25 = poi_inds_x[2]  # shading boundary = old 25 km point

In [ ]:
# Conversion of the old km values (straight-line axis) to along-line km
L = centerline[-1, 2] / 1000
old_x = centerline_straight.max() / 1000 - centerline_straight / 1000
new_x = L - centerline[:, 2] / 1000
rows = []
for k in [9.3, 23.3, 24, 33, 33.3, 38, 39, 39.3, 40, 58, 61.3, 62, 63.3, 64, 65, 65.3, 66, 75, 79.3]:
    i = int(np.argmin(np.abs(old_x - k)))
    rows.append((k, round(float(new_x[i]), 1)))
print('old km -> along-line km:', rows)
pd.DataFrame(rows, columns=['old_km_straight', 'new_km_along_line']).to_csv(f'{OUT}/km_conversion.csv', index=False)

## Zone means (Table 2) and transverse zones
Mean elevation change per zone on the 10 m Savitzky-Golay samples, using only samples finite in both profiles and in the 1995 profile (as plotted in Figs. 4-5). Writes `centerline_zones.csv`, `centerline_points_km.csv`, `balance_line_2007_2009_crossings.csv`, `transverse_zones.csv`.

In [ ]:
xc = centerline[:, 2].max() / 1000 - centerline[:, 2] / 1000          # km from the upper end
P = {k + 1: centerline[:, 2].max() / 1000 - p / 1000 for k, p in enumerate(poi)}
term = xc.max()
zones = {'acc (start-1)': (0, P[1]), 'trunk (1-4)': (P[1], P[4]), 'upper lobe (4-7)': (P[4], P[7]),
         'lower lobe (7-9)': (P[7], P[9]), 'lower lobe (7-term)': (P[7], term + 1e-6), '9-term': (P[9], term + 1e-6)}
H = {k: np.where(np.isfinite(centerline_smoothed[0]), v, np.nan) for k, v in {d.strftime('%Y-%m-%d'): centerline_smoothed[i] for i, d in enumerate(dates_oib)}.items()}
D = {d.strftime('%Y-%m-%d'): d for d in dates_oib}
fig_dates = ['1995-06-05'] + [dates_oib[i].strftime('%Y-%m-%d') for i in show]
assert fig_dates[0] == dates_oib[0].strftime('%Y-%m-%d'), dates_oib[0]

def zmean(a, b, z):
    lo, hi = zones[z]; s = (xc >= lo) & (xc < hi)
    both = s & np.isfinite(H[a]) & np.isfinite(H[b])
    return (np.mean(H[b][both] - H[a][both]) if both.any() else np.nan), int(both.sum()), 100 * both.sum() / s.sum()

zrows = []
for kind, pairs in [('vs2007', [('2007-08-26', d) for d in fig_dates]), ('vs1995', [('1995-06-05', d) for d in fig_dates]),
                    ('consecutive', list(zip(fig_dates[:-1], fig_dates[1:]))),
                    ('pair', [('1995-06-05', '2007-08-26'), ('2014-08-20', '2018-08-17'), ('2018-08-17', '2021-05-02')])]:
    for a, b in pairs:
        dt = (D[b] - D[a]).days / 365.25
        for z in zones:
            m, n, cov = zmean(a, b, z)
            zrows.append({'kind': kind, 'from': a, 'to': b, 'dt_a': dt, 'zone': z, 'dh_m': m, 'rate_m_a': m / dt if dt else np.nan, 'n': n, 'cov_pct': cov})
zdf = pd.DataFrame(zrows)
zdf.to_csv(f'{OUT}/centerline_zones.csv', index=False)
pd.DataFrame([{'point': k, 'km_from_upper_end': v} for k, v in P.items()] + [{'point': 'terminus', 'km_from_upper_end': term}]).to_csv(f'{OUT}/centerline_points_km.csv', index=False)

# Balance line: sign changes of the 2007->2009 difference between points 4 and 9
d79 = H['2009-08-22'] - H['2007-08-26']
o = np.argsort(xc); xs, ds = xc[o], d79[o]
s = (xs >= P[4]) & (xs <= P[9]) & np.isfinite(ds)
xs, ds = xs[s], ds[s]
cross = [(xs[j] + xs[j + 1]) / 2 for j in range(len(xs) - 1) if np.sign(ds[j]) != np.sign(ds[j + 1]) and xs[j + 1] - xs[j] < 0.05]
pd.DataFrame({'km': cross}).to_csv(f'{OUT}/balance_line_2007_2009_crossings.csv', index=False)
print('points', {k: round(v, 2) for k, v in P.items()}, 'terminus', round(term, 2))
print('2007->2009 sign changes between points 4 and 9 (km):', [round(c, 2) for c in cross])

# Transverse line, dh vs 1995 and the 2007->2010 difference
_ok95 = np.isfinite(cross_line_smoothed[0])
xx = cross_line[:, 2] / 1000
Px = {k + 1: v for k, v in enumerate(poi_x)}
HX = {d.strftime('%Y-%m-%d'): np.where(_ok95, cross_line_smoothed[i], np.nan) for i, d in enumerate(dates_oib)}
xz = {'west of 2': (0, Px[2]), '2-3': (Px[2], Px[3]), 'east of 4': (Px[4], xx.max() + 1e-6), 'west of 1': (0, Px[1])}
xrows = []
for i in show_x:
    b = dates_oib[i].strftime('%Y-%m-%d')
    for z, (lo, hi) in xz.items():
        sel = (xx >= lo) & (xx < hi); both = sel & np.isfinite(HX[b]) & np.isfinite(HX['1995-06-05'])
        xrows.append({'date': b, 'zone': z, 'dh_vs1995_m': np.mean(HX[b][both] - HX['1995-06-05'][both]) if both.any() else np.nan,
                      'n': int(both.sum()), 'cov_pct': 100 * both.sum() / sel.sum()})
dx = HX['2010-08-23'] - HX['2007-08-26']
for lab, (lo, hi) in {'2007->2010 point 2 to 30 km': (Px[2], 30), '2007->2010 west of point 1': (0, Px[1]),
                      '2007->2010 36-42 km': (36, 42), '2007->2010 point 2 to point 3': (Px[2], Px[3])}.items():
    sel = (xx >= lo) & (xx < hi) & np.isfinite(dx)
    v = dx[sel]
    xrows.append({'date': lab, 'zone': 'min/p5/median/p95/max', 'dh_vs1995_m': np.nan, 'n': int(sel.sum()),
                  'cov_pct': 100 * sel.sum() / ((xx >= lo) & (xx < hi)).sum(),
                  'stats': '/'.join(f'{q:+.1f}' for q in [v.min(), np.percentile(v, 5), np.median(v), np.percentile(v, 95), v.max()])})
xdf = pd.DataFrame(xrows)
xdf.to_csv(f'{OUT}/transverse_zones.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 500)
print(zdf.pivot_table(index=['kind', 'from', 'to'], columns='zone', values='dh_m', sort=False).round(1))
print(xdf.round(1).to_string(index=False))

## Transverse elevation-difference panel statistics
2007-08-26 -> 2010-08-23 and 2018-08-17 -> 2021-05-02 per sample (Fig. 5c), only where the 1995 profile is finite. Writes `crossline_dh_panel_medians.csv`.

In [ ]:
x_km = cross_line[:, 2] / 1000
max_x_km = cross_line[:, 2].max() / 1000
_dh_lines = {}
for _a, _b in [('2007-08-26', '2010-08-23'), ('2018-08-17', '2021-05-02')]:
    _ia, _ib = [dates_oib.index(pd.Timestamp(d)) for d in (_a, _b)]
    _dh_lines[(_a, _b)] = np.where(_ok95, cross_line_smoothed[_ib] - cross_line_smoothed[_ia], np.nan)

_P = {k + 1: v for k, v in enumerate(poi_x)}
_zones = {'west of point 1': (0, _P[1]), 'point 2 to 30 km': (_P[2], 30), 'points 2-3': (_P[2], _P[3]),
          'east of point 4': (_P[4], max_x_km + 1e-6), '36-42 km': (36, 42)}
_rows = []
for (_a, _b), _d in _dh_lines.items():
    for _z, (_lo, _hi) in _zones.items():
        _s = (x_km >= _lo) & (x_km < _hi)
        _v = _d[_s & np.isfinite(_d)]
        _rows.append({'pair': f'{_a} -> {_b}', 'zone': _z, 'km_from': _lo, 'km_to': min(_hi, max_x_km), 'n': _v.size,
                      'cov_pct': 100 * _v.size / _s.sum(), 'median_m': np.median(_v) if _v.size else np.nan,
                      'p25_m': np.percentile(_v, 25) if _v.size else np.nan, 'p75_m': np.percentile(_v, 75) if _v.size else np.nan,
                      'mean_m': _v.mean() if _v.size else np.nan})
_med = pd.DataFrame(_rows); _med.to_csv(f'{OUT}/crossline_dh_panel_medians.csv', index=False)
print('points (km):', {k: round(v, 2) for k, v in _P.items()})
print(_med.round(1).to_string(index=False))

## Table 2 (LaTeX)
Zone means relative to the August 2007 profile. Writes `table2_centerline_zones.tex`.

In [ ]:
v = zdf[zdf.kind == 'vs2007'].set_index(['to', 'zone']).dh_m
rows = ['2007-08-26', '2009-08-22', '2010-08-23', '2012-08-21', '2013-08-27', '2014-08-20', '2018-08-17', '2021-05-02']
lab = {'2018-08-17': '2018$^\\dagger$', '2021-05-02': 'May 2021$^\\dagger$'}
tex = [r'\begin{table}[ht]', r'\centering',
       r'\caption{Mean surface elevation change relative to the August 2007 profile (m) along the Seward centerline, by zone (Fig.~\ref{fig:centerlines}). Each value uses only samples present in both profiles. Accumulation area: start to point 1; trunk: points 1--4; upper lobe: points 4--7; lower lobe: points 7--9. $^\dagger$Partial coverage, common samples only.}',
       r'\label{tab:centerline_zones}', r'\begin{tabular}{lrrrr}', r'\toprule',
       r'Profile & Accumulation area & Trunk & Upper lobe & Lower lobe \\', r'\midrule']
for d in rows:
    vals = [v[(d, k)] for k in ['acc (start-1)', 'trunk (1-4)', 'upper lobe (4-7)', 'lower lobe (7-9)']]
    name = lab.get(d, pd.Timestamp(d).strftime('%Y'))
    tex.append(f'{name} & ' + ' & '.join('0.0' if abs(x) < 0.05 else f'${x:+.1f}$'.replace('$-', '$-') for x in vals) + r' \\')
tex += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
open(f'{OUT}/table2_centerline_zones.tex', 'w').write('\n'.join(tex) + '\n')
print('\n'.join(tex))

## Figure inputs (Figs. 4-5)
Profile arrays read by `figures/fig4_5_profiles.py`:
- `<line>_profiles.csv`: 10 m samples (UTM 7N x, y; distance along line; plotted km axis; bed; smoothed OIB surface per date `surf_<date>`; plotted $dh_{95}$ per shown date `dh95_<date>`)
- `<line>_velocity.csv`: 100 m samples, mean and std of the monthly velocities per period
- `profile_layout.json`: shown dates, coloured line segments and zone boundaries
- `rgi_outlines_map.geojson`: RGI outlines used to mask the bed on the map

In [ ]:
yrs = [((d - dates_oib[0]).days) / 365.25 for d in dates_oib]

def profile_table(line, smoothed, dhdt, bed_line, shown, km):
    df = pd.DataFrame({'x': line[:, 0], 'y': line[:, 1], 'dist_m': line[:, 2], 'km': km,
                       'bed_m': np.ma.asarray(bed_line, dtype=float).filled(np.nan)})
    for i, d in enumerate(dates_oib):
        df[f'surf_{d:%Y-%m-%d}'] = smoothed[i]
    for i in shown:
        df[f'dh95_{dates_oib[i]:%Y-%m-%d}'] = dhdt[i] * yrs[i]
    return df

def velocity_table(v, km):
    dts = pd.to_datetime(v.mid_date.values)
    df = pd.DataFrame({'km': km})
    for lab, m in [('2000-2012', dts < pd.Timestamp('2012-01-01')),
                   ('2012-2020', (dts >= pd.Timestamp('2012-01-01')) & (dts < pd.Timestamp('2020-01-01'))),
                   ('2020-2022', dts >= pd.Timestamp('2020-01-01'))]:
        d = v.values[m, :]
        if len(d) > 0:
            df[f'mean_{lab}'], df[f'std_{lab}'] = np.nanmean(d, axis=0), np.nanstd(d, axis=0)
    return df

profile_table(centerline, centerline_smoothed, dhdt_centerline, bed_centerline, show,
              max_dist_km - centerline[:, 2] / 1000).to_csv(f'{OUT}/centerline_profiles.csv', index=False, float_format='%.4f')
profile_table(cross_line, cross_line_smoothed, dhdt_cross_line, bed_cross_line, show_x,
              cross_line[:, 2] / 1000).to_csv(f'{OUT}/crossline_profiles.csv', index=False, float_format='%.4f')
velocity_table(v_centerline, max_dist_km - centerline_vels[:, 2] / 1000).to_csv(f'{OUT}/centerline_velocity.csv', index=False, float_format='%.4f')
velocity_table(v_crossline, crossline_vels[:, 2] / 1000).to_csv(f'{OUT}/crossline_velocity.csv', index=False, float_format='%.4f')

layout = {
    'centerline': {'shown': [dates_oib[i].strftime('%Y-%m-%d') for i in show], 'length_km': max_dist_km,
                   'ind_40': int(ind_40), 'ind_70': int(ind_70), 'b_lobe_km': b_lobe, 'b_trunk_km': b_trunk},
    'crossline': {'shown': [dates_oib[i].strftime('%Y-%m-%d') for i in show_x], 'length_km': max_x_km, 'ind_25': int(ind_25)},
}
json.dump(layout, open(f'{OUT}/profile_layout.json', 'w'), indent=1)

_rgi = gpd.read_file('../External_Data/RGI60/01_rgi60_Alaska.shp')
glaciers_of_interest = _rgi[_rgi['RGIId'].isin(['RGI60-01.13696', 'RGI60-01.23653', 'RGI60-01.23649'])]
glaciers_of_interest.to_file(f'{OUT}/rgi_outlines_map.geojson', driver='GeoJSON')